# Six simple encoding demonstrations: German Credit Data

**Purpose:** Show how text categories become numbers.

Place `German Credit Data(1).csv` in the same folder as this notebook, then run the cells from top to bottom. The dataset contains 1,000 credit records. Its category names are codes such as `A171` and `A172`; we will treat them simply as category names. `status` is already numeric: **0 = good credit, 1 = bad credit**.

**What to say first:** “A computer needs numbers to work with categories. There are several ways to represent a category as numbers. Let us see what each way produces.”

## Read the CSV
The first five rows help us see the original text values before encoding.

In [ ]:
import pandas as pd

df = pd.read_csv("German Credit Data.csv")
print("Rows and columns:", df.shape)
df.head()

## 1. Label encoding

**Meaning:** Give each `job` category one number. For example, `A171 → 0`, `A172 → 1`. The particular numbers are labels; they do **not** tell us that one job is better than another.

**When useful:** Especially convenient when the answer you want to predict is a text label. Use caution if the input categories have no natural order.

**Pros:** Very simple; adds only one column.

**Cons:** The numbers can incorrectly suggest an order. Future data must use the same mapping.

In [ ]:
job_codes = sorted(df["job"].unique())
job_mapping = {job: number for number, job in enumerate(job_codes)}

label_demo = df[["job"]].copy()
label_demo["job_number"] = label_demo["job"].map(job_mapping)
print("Mapping:", job_mapping)
label_demo.head(8)

## 2. Ordinal encoding

**Meaning:** Give numbers to categories that **really have an order**. We first create easy-to-read loan duration groups from the numeric `duration` column: **short, medium, long**. Then encode them as `short → 0`, `medium → 1`, `long → 2`.

**When useful:** A genuine ranking, such as low/medium/high or short/medium/long. Do not assume the unexplained A-codes are ordered.

**Pros:** Preserves the known order; one column.

**Cons:** The numbers may suggest equal distances between groups even when that is not true. We have simplified numeric duration into groups, so some detail is lost.

In [ ]:
ordinal_demo = df[["duration"]].copy()
ordinal_demo["duration_group"] = pd.cut(
    ordinal_demo["duration"],
    bins=[0, 12, 36, float("inf")],
    labels=["short", "medium", "long"]
)
duration_mapping = {"short": 0, "medium": 1, "long": 2}
ordinal_demo["duration_number"] = ordinal_demo["duration_group"].map(duration_mapping)
print("Mapping:", duration_mapping)
ordinal_demo.head(8)

## 3. One-hot encoding

**Meaning:** Make **one column per category**. For `job`, each row has a 1 in its own job column and 0 in the other job columns. Example: `A171` becomes `job_A171 = 1`.

**When useful:** Categories such as `job` that do not have a meaningful order.

**Pros:** Does not invent a ranking; very easy to show visually.

**Cons:** A feature with many categories creates many columns.

In [ ]:
one_hot_demo = pd.get_dummies(df["job"], prefix="job", dtype=int)
pd.concat([df[["job"]], one_hot_demo], axis=1).head(8)

## 4. Dummy encoding (`drop_first=True`)

**Meaning:** This is a **variation of one-hot encoding**. It makes 0/1 columns but leaves out one category. If all the shown columns are 0, the row belongs to the omitted **reference category**. This is the technique used by `pd.get_dummies(..., drop_first=True)` in your credit classification notebook.

**When useful:** When you want fewer columns and a reference category, especially for interpreting a simple regression model.

**Pros:** One fewer column per original categorical feature; no artificial ranking.

**Cons:** The missing reference category is less obvious to a beginner. It is not a fundamentally separate family from one-hot encoding.

In [ ]:
dummy_demo = pd.get_dummies(df["job"], prefix="job", drop_first=True, dtype=int)
print("Original job categories:", sorted(df["job"].unique()))
print("Columns kept:", list(dummy_demo.columns))
pd.concat([df[["job"]], dummy_demo], axis=1).head(8)

## 5. Frequency encoding

**Meaning:** Replace each category with its **count** in the data. If `job = A173` appears 630 times, every `A173` row gets the number 630.

**When useful:** When many categories exist and you want to keep just one column.

**Pros:** Simple; one column; shows which categories are common or rare.

**Cons:** Two different categories can have the same count. The count says nothing by itself about bad credit. In a real project, learn counts from training data and reuse them for new data.

In [ ]:
job_counts = df["job"].value_counts()
frequency_demo = df[["job"]].copy()
frequency_demo["job_count"] = frequency_demo["job"].map(job_counts)
print("Job counts:", job_counts.to_dict())
frequency_demo.head(8)

## 6. Target encoding

**Meaning:** Replace a job code with the share of its records whose `status` is 1 (**bad credit**). For example, if 20 out of 100 records for a job had status 1, its encoded value would be `0.20`.

**When useful:** Sometimes helpful when a category has many distinct values and is related to the outcome.

**Pros:** One column; shows the relationship with the outcome.

**Cons:** **Easy to misuse.** The calculation below is only to demonstrate the idea. It uses the whole dataset, so it must **not** be used as a trustworthy evaluation. For real modeling, first split the data, calculate rates using training data only, and use careful out-of-fold encoding within training. Small groups can give misleading rates.

In [ ]:
bad_credit_rate_by_job = df.groupby("job")["status"].mean()
target_demo = df[["job", "status"]].copy()
target_demo["job_bad_credit_rate_demo_only"] = target_demo["job"].map(bad_credit_rate_by_job)
print("Example bad-credit rates by job:")
print(bad_credit_rate_by_job.round(2))
target_demo.head(8)

## Recap: which technique should I demonstrate first?

| Technique | What learners see | Best simple example | Main drawback |
|---|---|---|---|
| Label | One integer per category | `job` codes | False order |
| Ordinal | Numbers in real order | Short/medium/long duration | Loses detail when grouping |
| One-hot | A 0/1 column for every category | `job` | More columns |
| Dummy | One-hot with one category omitted | `job` | Reference category is implicit |
| Frequency | How often each category occurs | `job` count | Different categories may share a count |
| Target | Bad-credit rate per category | `job` and `status` | Leakage if computed incorrectly |

